# rPPG — standart veri setleri arası protokoller ve tohum tekrarları

**Varsayılan: S2, seed 42 — UBFC ile sıfırdan eğitim, PURE üzerinde test.**
GPU ve Internet açık; mevcut `HF_TOKEN` ve `HF_WRITE_TOKEN` Secrets erişimleri açık olsun.
Save Version → Save & Run All. Her çalıştırma tek deney/tohumdur; oturum süresi üç eğitime bölünmez.

- `DENEY="S1"`: yalnızca PURE eğitim/doğrulama, yalnızca UBFC test.
- `DENEY="S2"`: yalnızca UBFC eğitim/doğrulama, yalnızca PURE test.
- Tekrarlar için `SEED=42`, `43`, `44` ile ayrı koşular yapın. Model başlangıcı ve örnekleme tohumu değişir;
  kişi bazındaki eğitim/doğrulama ayrımı bütün tohumlarda sabittir.
- Yeni çıktılar deney + tohumla ayrı HF klasöründedir; S1'in önceki tamamlanan çıktıları kullanılmaz.
- Bütün oturumlar hazır olmadan eğitim başlamaz. Hedef test kaynağı eğitim kökünde bulunmaz.
- PURE hedefken, PURE üzerinde önceden eğitilmiş hazır model kıyaslamaya eklenmez.
- Sonuçlar kendi ön işleme ve nabız hesaplama protokolümüze aittir; farklı yayın tablolarıyla doğrudan aynı ölçüm olduğu varsayılmaz.

Ham PURE kareleri işlendikten sonra geçici kopya silinir. Yeni oturumda ham veri yeniden indirilir.
Model/optimizer checkpoint'i 30 dakikada bir yedeklenir; aynı deney/tohumla tekrar çalıştırmak eğitimi sürdürür.


In [ ]:
# Yalnızca bu iki ayarı değiştirin: her tohum için ayrı Save & Run All.
DENEY = "S2"
SEED = 42
assert DENEY in ("S1", "S2") and isinstance(SEED, int) and SEED > 0
TRAIN_SOURCE, TEST_SOURCE = ("pure", "ubfc") if DENEY == "S1" else ("ubfc", "pure")
AD = f"MMRPhys {DENEY}, seed {SEED}: {TRAIN_SOURCE} -> {TEST_SOURCE}"
TRAIN_ARGS = (f"--arch mmrphys --scratch --lr 1e-3 --val-max 40 "
              f"--source-weights {TRAIN_SOURCE}=1 --seed {SEED}")
RUN = f"standart_{DENEY.lower()}_seed{SEED}_v1"
STD = True
EPOCHS, STEPS = 30, 500
SADECE_DEGERLENDIRME = False
TUR7E = "results/derin_ogrenme/kaggle_tur7_E/last.pth"


In [ ]:
# 1) Ortam, kod (rppg, rPPG-Toolbox, MMRPhys sabit sürüm), anahtarlar
import os, subprocess, shutil, sys, time, glob
from datetime import datetime, timedelta

# PyTorch Kaggle ortamından kullanılır; CUDA kurulumuna dokunulmaz.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "huggingface_hub>=0.27,<2", "opencv-python-headless>=4.8,<5",
                "scipy", "pandas", "scikit-learn", "matplotlib", "tabulate", "psutil", "einops"], check=True)
T0 = datetime.now()
# Hugging Face: xet parça önbelleği diske ~10 GB yazabiliyor, ilerleme çubukları günlükleri şişiriyor
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
print("ortam:", "Kaggle" if KAGGLE else "Colab")
if shutil.which("nvidia-smi"):   # GPU'suz oturumda komut yok (yalnızca değerlendirme)
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi") or SADECE_DEGERLENDIRME, ("GPU yok: Settings -> Accelerator: GPU T4 x2" if KAGGLE else
                                    "GPU yok: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU")
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"

REPO = os.path.join(WORK, "rppg")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
sys.path.insert(0, os.path.join(REPO, "scripts"))
from dl_common import MMRPHYS, MMRPHYS_COMMIT
if not os.path.exists(MMRPHYS):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/PhysiologicAILab/MMRPhys.git", MMRPHYS], check=True)
    head = subprocess.run(["git", "-C", MMRPHYS, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    if head != MMRPHYS_COMMIT:   # depo güncellendiyse yerelde denenen sürüme dön
        subprocess.run(["git", "-C", MMRPHYS, "fetch", "--depth", "1", "origin", MMRPHYS_COMMIT], check=True)
        subprocess.run(["git", "-C", MMRPHYS, "checkout", MMRPHYS_COMMIT], check=True)
print("MMRPhys", subprocess.run(["git", "-C", MMRPHYS, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

def secret(name):
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)

for name in ("HF_TOKEN", "HF_WRITE_TOKEN"):
    try:
        os.environ[name] = secret(name)
        print(f"{name} var")
    except Exception:
        raise SystemExit(f"{name} yok: " + ("Add-ons -> Secrets -> kutusunu işaretleyip defteri yeniden başlatın" if KAGGLE
                         else "soldaki 🔑 Gizli anahtarlar -> ekleyip 'Not defteri erişimi'ni açın"))

import psutil
from huggingface_hub import HfApi, snapshot_download
API = HfApi(token=os.environ["HF_WRITE_TOKEN"])
OUT_REPO = f"{API.whoami()['name']}/rppg-cikti"
DATA, RAW = "/tmp/dl", "/tmp/dl_raw"
OUT = os.path.join(WORK, "cikti")
FREE0 = shutil.disk_usage("/tmp").free / 1e9   # Colab'da df gerçek; Kaggle'da değil (bkz. disk notu)
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, LOGS):
    os.makedirs(d, exist_ok=True)
TUR5 = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur5/last.pth")   # hizalama referansı, karşılaştırma
TUR3 = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur3/last.pth")   # uygulamadaki model
TUR7B = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur7_B/best.pth")  # MMRPhys sıfırdan
TUR7E = os.path.join(REPO, TUR7E)  # 8. turun başlangıcı ve karşılaştırma tabanı
TRAIN_UNTIL = T0 + timedelta(hours=10)          # değerlendirme ve son yedek için pay
print(f"deney {DENEY}: {AD} | {TRAIN_ARGS} | disk /tmp boş {FREE0:.0f} GB"
      f" | RAM {psutil.virtual_memory().total / 1e9:.0f} GB | eğitim en geç {TRAIN_UNTIL:%H:%M}")

In [ ]:
# 2) Önceki oturumdan devam: Hugging Face'teki yedek (last.pth, best.pth, gecmis.json)
import threading
_bk = None
def _upload(note):
    try:
        API.create_repo(OUT_REPO, repo_type="dataset", private=True, exist_ok=True)
        API.upload_folder(folder_path=OUT, repo_id=OUT_REPO, repo_type="dataset", path_in_repo=RUN,
                          commit_message=f"{RUN}: {note}")
        print(f"yedeklendi ({note}): huggingface.co/datasets/{OUT_REPO}/tree/main/{RUN}", flush=True)
    except Exception as e:
        print("yedekleme hatası:", repr(e), flush=True)

def backup(note, wait=0):
    """Arka planda yükler: ağ takılsa da izleme döngüsü durmaz (7. tur B'de 22:31'den sonra yedek gelmedi).
    wait > 0: en çok o kadar saniye bekler (son yedekler)."""
    global _bk
    if _bk is not None and _bk.is_alive():
        print("önceki yedekleme sürüyor, bu tur atlandı", flush=True)
    else:
        _bk = threading.Thread(target=_upload, args=(note,), daemon=True)
        _bk.start()
    if wait:
        _bk.join(wait)

try:
    snapshot_download(OUT_REPO, repo_type="dataset", local_dir="/tmp/onceki", token=API.token,
                      allow_patterns=[f"{RUN}/last.pth", f"{RUN}/best.pth", f"{RUN}/gecmis.json"])
    for f in glob.glob(f"/tmp/onceki/{RUN}/*"):
        shutil.copy(f, OUT)
    print("önceki oturum:", sorted(os.listdir(OUT)))
except Exception as e:
    print("önceki oturum yok, baştan:", type(e).__name__)

In [ ]:
# PURE: önce tek oturumla gerçek erişim ve ön işleme kontrolü, sonra tamamı.
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")
def run(cmd, name):
    path = os.path.join(LOGS, name + ".log")
    print(f"{name} başladı; günlük: {path}", flush=True)
    with open(path, "a", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, encoding="utf-8", errors="replace", env=env)
        for line in p.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        rc = p.wait()
    if rc:
        raise RuntimeError(f"{name} başarısız (kod {rc}); günlük: {path}")
    return rc

pure_cmd = [PY, "-u", "scripts/dl_prepare_pure.py", "--out", DATA, "--raw", RAW, "--workers", "4"]
run([*pure_cmd, "--limit", "1"], "pure_ilk_oturum")
run(pure_cmd, "pure_tamami")


In [ ]:
# UBFC'nin 42 kişisini tamamla; protokole göre yalnızca eğitim veya yalnızca test olur.
import json, zlib
from pathlib import Path
import numpy as np
from dl_prepare import UBFC_TEST, job_test_dl, run as prepare_jobs
ubfc_jobs = [(job_test_dl, ("ubfc", s, RAW, DATA, "test")) for s in sorted(UBFC_TEST)]
if prepare_jobs(ubfc_jobs, 2):
    raise RuntimeError("UBFC verisi hazırlanamadı")
run([PY, "-u", "scripts/dl_prepare.py", "--out", DATA, "--raw", RAW,
     "--ubfc", "34", "--workers", "1"], "ubfc_kalan34")

pure_files = sorted(Path(DATA, "pure").glob("*.npz"))
ubfc_files = sorted(Path(DATA, "ubfc").glob("*.npz")) + sorted(Path(DATA, "test").glob("ubfc_*.npz"))
assert len(ubfc_files) == 42, f"UBFC eksik: {len(ubfc_files)}/42"
pure_manifest = json.loads(Path(DATA, "pure", "manifest.json").read_text())
assert len(pure_files) == len(pure_manifest["sessions"]), "PURE oturumları eksik"
training_files, testing_files = (pure_files, ubfc_files) if DENEY == "S1" else (ubfc_files, pure_files)
TRAIN_DATA = os.path.join("/tmp", RUN + "_train")
TEST_DATA = os.path.join("/tmp", RUN + "_test")
train_dir = Path(TRAIN_DATA, TRAIN_SOURCE)
train_dir.mkdir(parents=True, exist_ok=True)
Path(TEST_DATA).mkdir(parents=True, exist_ok=True)

def link_files(files, dest, strip_ubfc=False):
    for p in files:
        name = p.stem.removeprefix("ubfc_") if strip_ubfc else p.stem
        for ext in (".npy", ".npz"):
            src = p.with_suffix(ext)
            dst = Path(dest, name + ext)
            if src.exists() and not dst.exists():
                dst.symlink_to(src)

link_files(training_files, train_dir, strip_ubfc=TRAIN_SOURCE == "ubfc")
link_files(testing_files, TEST_DATA, strip_ubfc=TEST_SOURCE == "ubfc")
assert {p.name for p in Path(TRAIN_DATA).iterdir()} == {TRAIN_SOURCE}
train_people, val_people, test_people = set(), set(), set()
for files, target in ((training_files, None), (testing_files, test_people)):
    for p in files:
        with np.load(p) as m:
            assert str(m["source"]) == (TEST_SOURCE if target is not None else TRAIN_SOURCE)
            person = str(m["person"])
            if target is not None:
                target.add(person)
            elif zlib.crc32(person.encode()) % 10 == 0:
                val_people.add(person)
            else:
                train_people.add(person)
assert train_people and val_people and test_people
assert not (train_people & val_people or train_people & test_people or val_people & test_people)
manifest = {"experiment": DENEY, "seed": SEED, "train_source": TRAIN_SOURCE, "test_source": TEST_SOURCE,
            "train_people": sorted(train_people), "validation_people": sorted(val_people),
            "test_people": sorted(test_people), "training_input_sessions": len(training_files),
            "test_input_sessions": len(testing_files), "pure_source": pure_manifest,
            "quality_filter": "face_rate >= 0.5; eğitimde ek etiket kalite filtresi"}
Path(OUT, "protokol.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
# Aynı RUN altında farklı ayarlardan checkpoint kullanılmasını engelle.
history = Path(OUT, "gecmis.json")
if history.exists():
    settings = json.loads(history.read_text())["ayarlar"]
    assert settings["seed"] == SEED and settings["source_weights"] == f"{TRAIN_SOURCE}=1"
jobs = {}
backup("veri ve kişi ayrımı doğrulandı")
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Eğitim: bütün oturumlar hazır; yalnızca seçilen kaynak eğitim ve doğrulamada.
SUBS = ("mcd", "ubfc", "pure", "dlcn", "vipl", "cohface", "ubfcphys")
TESTS = ("test", "test_kamera", "test_vipl", "test_mmpd", "test_dlcn", "test_cohface", "test_ubfcphys")
def counts(subs=SUBS):
    return {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in subs}

while sum(counts().values()) < (40 if STD else 300) and any(j.poll() is None for j in jobs.values()):
    print(time.strftime("%H:%M"), counts(), flush=True)
    time.sleep(120)

if datetime.now() >= TRAIN_UNTIL:
    raise RuntimeError("Veri hazırlama oturum süresini doldurdu; yeni oturumda yeniden başlatın")
train = subprocess.Popen(
    [PY, "-u", "scripts/dl_train.py", "--data", TRAIN_DATA, "--out", OUT, *TRAIN_ARGS.split(), "--epochs", str(EPOCHS),
     "--steps", str(STEPS), "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")],
    cwd=REPO, env=env, stdout=open(os.path.join(LOGS, "egitim.log"), "a"), stderr=subprocess.STDOUT)
print("eğitim başladı:", " ".join(train.args[2:]))

In [ ]:
# 6) İzleme: 5 dakikada bir özet (gunlukler/izleme.log'a da yazılır ve yedeklenir), 30 dakikada bir yedek.
#    Bellek koruması: boş RAM 4 GB'ın altına inerse veri hazırlama duraklatılır, 8 GB'ı geçince sürer.
#    (6. tur ve 7. tur B hata dökümü olmadan durdu; Kaggle başarısız sürümün çıktısını saklamıyor.)
import signal
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

def tree_rss(pid):
    try:
        p = psutil.Process(pid)
        return p.memory_info().rss + sum(c.memory_info().rss for c in p.children(recursive=True))
    except Exception:
        return 0

def signal_tree(pid, sig):
    try:
        p = psutil.Process(pid)
        for q in [p] + p.children(recursive=True):
            q.send_signal(sig)
    except Exception:
        pass

def log_line(msg):
    print(msg, flush=True)
    with open(os.path.join(LOGS, "izleme.log"), "a", encoding="utf-8") as f:
        f.write(msg + "\n")

DISK_LIMIT_GB = 97 if KAGGLE else max(20, FREE0 - 10)   # Colab: başlangıçtaki boş alan - 10 GB pay
def used_gb():
    """Bu oturumun yazdığı veri (df değil: Kaggle'da df gerçek sınırı göstermiyor)."""
    tot = 0
    for root in (DATA, RAW, "/tmp/onceki", os.path.expanduser("~/.cache/huggingface")):
        for d, _, fs in os.walk(root):
            for f in fs:
                try:
                    tot += os.path.getsize(os.path.join(d, f))
                except OSError:
                    pass
    return tot / 1e9

paused, last_backup, disk_stop = False, time.time(), False
while train.poll() is None:
    try:
        vm = psutil.virtual_memory()
        avail = vm.available / 1e9
        live = {k: j for k, j in jobs.items() if j.poll() is None}
        if avail < 4 and live and not paused:
            for j in live.values():
                signal_tree(j.pid, signal.SIGSTOP)
            paused = True
            log_line(f"{time.strftime('%H:%M')} UYARI: boş RAM {avail:.1f} GB, veri hazırlama duraklatıldı")
        elif paused and avail > 8:
            for j in live.values():
                signal_tree(j.pid, signal.SIGCONT)
            paused = False
            log_line(f"{time.strftime('%H:%M')} boş RAM {avail:.1f} GB, veri hazırlama sürüyor")
        used = used_gb()
        if used > DISK_LIMIT_GB and live and not disk_stop:
            for j in live.values():
                signal_tree(j.pid, signal.SIGCONT)
                subprocess.run(["pkill", "-TERM", "-P", str(j.pid)])
                j.terminate()
            disk_stop = True
            log_line(f"{time.strftime('%H:%M')} DİSK SINIRI: oturum verisi {used:.0f} GB > {DISK_LIMIT_GB:.0f} GB, "
                     "veri hazırlama durduruldu (eğitim eldeki veriyle sürüyor)")
            live = {}
        mem = " ".join(f"{k.removeprefix('hazirla_')} {tree_rss(j.pid) / 1e9:.1f}" for k, j in live.items())
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],  # defterde CUDA açılmasın
                             capture_output=True, text=True).stdout.strip().replace("\n", ", ") if shutil.which("nvidia-smi") else "yok"
        ep = [l for l in tail(os.path.join(LOGS, "egitim.log"), 400) if l.startswith("epoch ")]
        log_line(f"{time.strftime('%H:%M')} | {counts()} | test {counts(TESTS)}"
                 f" | veri {used:.0f}/{DISK_LIMIT_GB:.0f} GB, boş {shutil.disk_usage(WORK).free/1e9:.0f} GB"
                 f" | RAM boş {avail:.1f}/{vm.total/1e9:.0f} GB, eğitim {tree_rss(train.pid)/1e9:.1f} GB, {mem or 'hazırlama bitti'}"
                 f"{' (DURAKLATILDI)' if paused else ''} | GPU kullanılan {gpu}"
                 f" | {ep[-1] if ep else 'ilk epoch sürüyor'}")
    except Exception as e:
        print("izleme hatası:", repr(e), flush=True)
    if time.time() - last_backup > 1800:
        backup(f"eğitim sürüyor {time.strftime('%H:%M')}")
        last_backup = time.time()
    time.sleep(300)
if paused:
    for j in jobs.values():
        signal_tree(j.pid, signal.SIGCONT)
print("\n".join(tail(os.path.join(LOGS, "egitim.log"), 20)))
print("eğitim çıkış kodu:", train.returncode)
backup("eğitim bitti", wait=60)
if train.returncode:
    raise RuntimeError("Eğitim başarısız; egitim.log dosyasını inceleyin")

In [ ]:
# Hedef veri setinin tamamında değerlendirme; hazır PURE modeli S2'de kullanılmaz.
assert os.path.exists(os.path.join(OUT, "last.pth")), "Eğitim checkpoint'i yok"
cmd = [PY, "-u", "scripts/dl_evaluate.py", "--model", os.path.join(OUT, "last.pth"),
       "--name", AD, "--test", TEST_DATA, "--data", TRAIN_DATA,
       "--out", os.path.join(OUT, "test_" + TEST_SOURCE), "--prev", "yok.csv",
       "--history", os.path.join(OUT, "gecmis.json")]
if DENEY == "S2":
    cmd.append("--no-pretrained")
cmd.append("--compare")
if os.path.exists(os.path.join(OUT, "best.pth")):
    cmd.append(f"En iyi epoch={os.path.join(OUT, 'best.pth')}")
try:
    run(cmd, "degerlendirme_" + TEST_SOURCE)
finally:
    backup("değerlendirme çıktıları", wait=60)


In [ ]:
# 8) Çıktılar
for p in sorted(glob.glob(os.path.join(OUT, "**"), recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}")
os.chdir(WORK)
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
if not KAGGLE:   # Colab: soldaki Dosyalar panelinden indirilir (sonuçlar ayrıca Hugging Face'te)
    print("indirmek için:", shutil.make_archive(os.path.join(WORK, f"cikti_{RUN}"), "zip", OUT))
print("süre:", datetime.now() - T0)